# Connect to Drive

In [ ]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists("src"):
        !git clone https://github.com/HenriqueSchmitz/world-models-implementation temp_repo
        !mv temp_repo/* .
        !rm -rf temp_repo
        %pip install -q -r requirements.txt

In [ ]:
import os
from src.utils.colab import is_environment_colab_notebook
from src.utils.secrets import get_secret

project_folder = get_secret("worldModelsFolderPath") if is_environment_colab_notebook() else "./"
settings_path = os.path.join(project_folder, "settings.json")
world_model_folder = os.path.join(project_folder, "weights/worldmodel")
output_folder = os.path.join(project_folder, "weights/controller")
os.makedirs(output_folder, exist_ok=True)

# Training Settings

In [ ]:
from src.utils.logging import get_logger
LOG_LEVEL = "INFO"
logger = get_logger(LOG_LEVEL)

In [ ]:
import json

with open(settings_path, "r") as settings_file:
    settings = json.load(settings_file)
    REPRESENTATION_DIM = settings["vae"]["model"]["representation_dim"]
    RNN_HIDDEN_DIM = settings["world_model"]["model"]["hidden_dim"]
    ACTION_DIM = settings["controller"]["model"]["action_dim"]

    NUM_GENERATIONS = settings["controller"]["train"]["num_generations"]
    POPULATION_SIZE = settings["controller"]["train"]["population_size"]
    STEPS_PER_ROLLOUT = settings["controller"]["train"]["steps_per_rollout"]
    ROLLOUTS_PER_SOLUTION = settings["controller"]["train"]["rollouts_per_solution"]
    SIGMA_INIT = settings["controller"]["train"]["sigma_init"]

WANDB_PROJECT = "world-models-paper"
WANDB_RUN_NAME = "controller"

def log_settings():
    logger.info("REPRESENTATION_DIM: ", REPRESENTATION_DIM)
    logger.info("RNN_HIDDEN_DIM: ", RNN_HIDDEN_DIM)
    logger.info("ACTION_DIM: ", ACTION_DIM)
    logger.info("========================================")
    logger.info("NUM_GENERATIONS: ", NUM_GENERATIONS)
    logger.info("POPULATION_SIZE: ", POPULATION_SIZE)
    logger.info("STEPS_PER_ROLLOUT: ", STEPS_PER_ROLLOUT)
    logger.info("ROLLOUTS_PER_SOLUTION: ", ROLLOUTS_PER_SOLUTION)
    logger.info("SIGMA_INIT: ", SIGMA_INIT)
    logger.info("========================================")
    logger.info("WANDB_PROJECT: ", WANDB_PROJECT)
    logger.info("WANDB_RUN_NAME: ", WANDB_RUN_NAME)

log_settings()

# Setup

In [ ]:
from src.models.controller import Controller
from src.models.simulation_worldmodel import SimulationWorldModel
from src.training.controller_trainer import ControllerEvolutionaryTrainer
from src.utils.torch import get_device
from src.utils.secrets import get_secret

In [6]:
DEVICE = get_device()

2025-12-10 08:32:37 [INFO] Logger initialized.
2025-12-10 08:32:37 [INFO] Using device: mps:0


# Train

In [ ]:
world_model_path = os.path.join(world_model_folder, f"model.pth")
simulation_worldmodel = SimulationWorldModel(worldmodel_path=world_model_path,
                                             settings_path=settings_path,
                                             device=DEVICE,
                                             batch_size=ROLLOUTS_PER_SOLUTION,
                                             logger=logger)

In [ ]:
model = Controller(observation_dim=REPRESENTATION_DIM,
                   hidden_dim=RNN_HIDDEN_DIM,
                   action_dim=ACTION_DIM,
                   device=DEVICE)

In [ ]:
wandb_setup = {
    "api_key": get_secret('wandbApiKey'),
    "project": WANDB_PROJECT,
    "run_name": WANDB_RUN_NAME,
    "config": {
        "generations": NUM_GENERATIONS,
        "population_size": POPULATION_SIZE,
        "steps_per_rollout": STEPS_PER_ROLLOUT,
        "rollouts_per_solution": ROLLOUTS_PER_SOLUTION,
        "sigma_init": SIGMA_INIT,
    }
}

In [10]:
trainer = ControllerEvolutionaryTrainer(model=model,
                                        weights_folder=output_folder,
                                        num_generations=NUM_GENERATIONS,
                                        population_size=POPULATION_SIZE,
                                        simulation_world_model=simulation_worldmodel,
                                        steps_per_rollout=STEPS_PER_ROLLOUT,
                                        rollouts_per_solution=ROLLOUTS_PER_SOLUTION,
                                        sigma_init=SIGMA_INIT,
                                        load_checkpoint=True,
                                        device=DEVICE,
                                        wandb_setup=wandb_setup,
                                        logger=logger)
                                        

2025-12-10 08:32:37 [INFO] Resuming training from: ./weights/controller/epoch_346.pth
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/henriqueschmitz/.netrc
wandb: Currently logged in as: schhenrique (schhenrique-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


(32_w,64)-aCMA-ES (mu_w=17.6,w_1=11%) in dimension 867 (seed=483228, Wed Dec 10 08:32:39 2025)


wandb: WARNING Tried to log to step 22182 that is less than the current step 22183. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


In [ ]:
# Repeat logging of important information so it is available on wandb run
log_settings()
get_device(logger)

In [ ]:
trainer.train()

Epoch:   0%|          | 0/500 [00:00<?, ?epoch/s]

Train Epoch 347:   0%|          | 0/64 [00:00<?, ?batch/s]

2025-12-10 08:34:31 [INFO] Epoch 347 Training Average Reward: 755.0355
2025-12-10 08:34:31 [INFO] Epoch 347 Training Max Reward: 5495.1816
2025-12-10 08:36:21 [INFO] Epoch 348 Training Average Reward: 752.2577
2025-12-10 08:36:21 [INFO] Epoch 348 Training Max Reward: 4196.0747
2025-12-10 08:38:09 [INFO] Epoch 349 Training Average Reward: 1442.7796
2025-12-10 08:38:09 [INFO] Epoch 349 Training Max Reward: 4934.0356
2025-12-10 08:39:58 [INFO] Epoch 350 Training Average Reward: 1080.1787
2025-12-10 08:39:58 [INFO] Epoch 350 Training Max Reward: 4163.9893
2025-12-10 08:41:47 [INFO] Epoch 351 Training Average Reward: 1244.7788
2025-12-10 08:41:47 [INFO] Epoch 351 Training Max Reward: 4263.1162
2025-12-10 08:43:34 [INFO] Epoch 352 Training Average Reward: 978.2580
2025-12-10 08:43:34 [INFO] Epoch 352 Training Max Reward: 5926.8115
2025-12-10 08:45:24 [INFO] Epoch 353 Training Average Reward: 1511.2613
2025-12-10 08:45:24 [INFO] Epoch 353 Training Max Reward: 5374.4385
2025-12-10 08:47:12 [IN

KeyboardInterrupt: 

socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x16cbe63c0>> (for post_run_cell), with arguments args (<ExecutionResult object at 314c61fd0, execution_count=11 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 175b4f690, raw_cell="trainer.train()" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/henriqueschmitz/Git/world-models-implementation/07-Train_Controller.ipynb#X45sZmlsZQ%3D%3D> result=None>,),kwargs {}:


socket.send() raised exception.


ConnectionResetError: Connection lost

socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.


socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.s